# 13. NER — Cibles discursives et score d'humanisation

AJOUT TÂCHE B1. Analyse des entités nommées (PER, ORG, GPE) par bloc, et score d'humanisation lexical (termes humanisants vs institutionnels-militaires).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import CORPUS_V3, BLOC_ORDER, BLOC_COLORS, BATCHES, month_to_batch
from ner_analysis import run_ner, extract_entity_counts, compute_humanization_score

FIG_DIR = Path("../figures")
RES_DIR = Path("../data/results")
FIG_DIR.mkdir(exist_ok=True)

def save(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()

## Chargement et NER

In [ ]:
df = pd.read_parquet(CORPUS_V3)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M").astype(str)
df["batch"] = df["month"].apply(month_to_batch)
df = df[df["bloc"].isin(BLOC_ORDER)]

text_col = "text_clean" if "text_clean" in df.columns else "text"
df[text_col] = df[text_col].fillna("").astype(str)

try:
    df = run_ner(df, text_col=text_col, model="fr_core_news_lg")
    print("NER avec spacy fr_core_news_lg")
except Exception as e:
    df = run_ner(df, text_col=text_col)
    print(f"NER fallback (spacy non dispo: {e})")

df["humanization_score"] = compute_humanization_score(df, text_col)
print(f"Corpus : {len(df)} textes, humanization_score : {df['humanization_score'].notna().sum()} non-null")

## Fig69 — Top-20 entités (PER+ORG+GPE) par bloc

In [ ]:
combined = []
for et in ["PER", "ORG", "GPE"]:
    col = f"entities_{et}"
    if col in df.columns:
        for val in df[col].dropna():
            try:
                import json
                lst = json.loads(val) if isinstance(val, str) else val
                if isinstance(lst, list):
                    combined.extend([(e, et) for e in lst])
            except Exception:
                pass

from collections import Counter
cnt_all = Counter(e for e, _ in combined)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, bloc in enumerate(BLOC_ORDER[:4]):
    sub = df[df["bloc"] == bloc]
    cnt = Counter()
    for et in ["PER", "ORG", "GPE"]:
        col = f"entities_{et}"
        if col in sub.columns:
            for val in sub[col].dropna():
                try:
                    import json
                    lst = json.loads(val) if isinstance(val, str) else val
                    if isinstance(lst, list):
                        cnt.update(lst)
                except Exception:
                    pass
    top20 = cnt.most_common(20)
    if top20:
        labels = [x[0][:20] for x in top20]
        vals = [x[1] for x in top20]
        axes[i].barh(range(len(labels)), vals, color=BLOC_COLORS.get(bloc, "#888"))
        axes[i].set_yticks(range(len(labels)))
        axes[i].set_yticklabels(labels, fontsize=9)
    axes[i].set_title(bloc)
    axes[i].set_xlabel("Frequence")
    axes[i].invert_yaxis()
plt.suptitle("Top-20 entites (PER+ORG+GPE) par bloc")
plt.tight_layout()
save("fig69_ner_entites_par_bloc")

## Fig70 — Humanization score par bloc × batch

In [ ]:
agg = df.dropna(subset=["humanization_score"]).groupby(["bloc", "batch"])["humanization_score"].mean().reset_index()
batch_order = [b for b in BATCHES.keys() if b in agg["batch"].unique()]
pivot = agg.pivot(index="bloc", columns="batch", values="humanization_score").reindex(BLOC_ORDER)
pivot = pivot[[c for c in batch_order if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(12, 6))
pivot.T.plot(kind="bar", ax=ax, color=[BLOC_COLORS.get(b, "#888") for b in BLOC_ORDER], width=0.8)
ax.set_ylabel("Humanization score (0-1)")
ax.set_xlabel("Batch")
ax.legend(title="Bloc", bbox_to_anchor=(1.02, 1))
ax.set_title("Score humanisation par bloc × batch")
plt.xticks(rotation=25, ha="right")
save("fig70_ner_humanization_batch")

## Fig71 — Évolution temporelle humanization_score par bloc

In [ ]:
monthly = df.dropna(subset=["humanization_score"]).groupby(["month", "bloc"]).agg(
    mean=("humanization_score", "mean"),
    sem=("humanization_score", "sem"),
    count=("humanization_score", "count")
).reset_index()
monthly = monthly[monthly["count"] >= 5]
monthly["month_ts"] = pd.to_datetime(monthly["month"] + "-01")

fig, ax = plt.subplots(figsize=(14, 6))
for bloc in BLOC_ORDER:
    sub = monthly[monthly["bloc"] == bloc]
    if len(sub) > 0:
        ax.plot(sub["month_ts"], sub["mean"], label=bloc, color=BLOC_COLORS.get(bloc, "#888"), lw=2)
        se = sub["sem"].fillna(0)
        ax.fill_between(sub["month_ts"], sub["mean"] - 1.96*se, sub["mean"] + 1.96*se, alpha=0.2, color=BLOC_COLORS.get(bloc, "#888"))
ax.set_ylabel("Humanization score (IC 95%)")
ax.legend()
ax.set_title("Evolution temporelle du score humanisation par bloc")
plt.xticks(rotation=25, ha="right")
save("fig71_ner_humanization_temporal")

## Interprétation

Quel bloc humanise le plus ? Correlation avec stance ?

In [ ]:
by_bloc = df.dropna(subset=["humanization_score", "stance_v3"]).groupby("bloc").agg(
    humanization_mean=("humanization_score", "mean"),
    stance_mean=("stance_v3", "mean")
).reindex(BLOC_ORDER)
print(by_bloc.to_string())
r = df["humanization_score"].corr(df["stance_v3"])
print(f"\nCorrelation humanization vs stance : r = {r:.3f}")